# Get all the EA Games On Steam

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

EA_URL = 'https://store.steampowered.com/search/?publisher=Electronic%20Arts&page='
# The list of apps which the scraped API does not have an official Steam store name
list_to_drop = [977660, 988670, 1379260, 1262591, 1315480, 1315442, 988662, 1315435, 1315471, 1315434, 1315470, 1315430, 
                1315431, 1315443, 1315472, 1315441, 1315440, 1315432, 1315433, 1363591, 1262590]

def scrape_ea_games(page):
    url = EA_URL + str(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    game_list = []
    
    games = soup.find_all('a', {'class': 'search_result_row'})
    for game in games:
        game_url = game['href']
        
        try:
            appid = game_url.split('/app/')[1].split('/')[0]
            
            # Ensure appid is numeric and a valid game
            if appid.isdigit():
                game_name = game.find('span', {'class': 'title'}).text.strip()
                game_list.append({'appid': appid, 'name': game_name})
        
        except IndexError:
            # In case the URL structure is unexpected, skip this game
            continue
    
    return game_list

def get_steam_app_details(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    try:
        response = requests.get(url)
        data = response.json()
        
        if data[str(app_id)]['success']:
            app_data = data[str(app_id)]['data']
            return app_data.get('type', 'unknown')  # Return the type of the app (game, dlc, etc.)
        else:
            return None
    except Exception as e:
        print(f"Error fetching data for app id {app_id}: {e}")
        return None

def collect_ea_games():
    all_games = []
    page = 1
    
    while True:
        print(f"Scraping page {page}...")
        games = scrape_ea_games(page)
        
        if not games:
            print("No more games found, stopping.")
            break
        
        all_games.extend(games)
        time.sleep(1)
        page += 1
    
    return all_games

# Collect EA games
ea_games = collect_ea_games()
print(f"Total EA games found: {len(ea_games)}")

# Separate games and DLCs
appid = []
dlc_id = []

for game in ea_games:
    print(game['appid'], game['name'])
    
    if game['appid'] not in str(list_to_drop):
        app_type = get_steam_app_details(game['appid'])  # Check if it's a game or DLC
        
        if app_type == 'game':
            appid.append(game['appid'])
        elif app_type == 'dlc':
            dlc_id.append(game['appid'])

# Output the results
print(f"Total games found: {len(appid)}")
print(f"Total DLCs found: {len(dlc_id)}")


# Write info on the designated games out to a csv file

In [ ]:
import requests
import time
from urllib.parse import quote

def get_steam_reviews(app_id, pages=100):
    base_reviews_url = f"https://store.steampowered.com/appreviews/{app_id}?json=1&num_per_page=20"
    reviews = []
    cursor = '*'  # Start with the first batch using "*"
    seen_cursors = set()  # To track and handle repeating cursors

    for page in range(1, pages + 1):
        print(f"Fetching page {page} for app ID {app_id}...")

        # URL-encode the cursor, replacing '+' with '%2B' if necessary
        encoded_cursor = quote(cursor).replace('+', '%2B')
        response = requests.get(f"{base_reviews_url}&cursor={encoded_cursor}")
        data = response.json()

        # Stop if no reviews are found
        if not data.get('reviews'):
            print("No more reviews found. Ending pagination.")
            break

        # Process each review
        for review in data['reviews']:
            reviews.append({
                'recommended': 'Recommended' if review.get('voted_up', False) else 'Not Recommended',
                'date': time.strftime('%Y-%m-%d', time.localtime(review.get('timestamp_created'))),
                'comment': review.get('review', 'N/A'),
                'playtime_hours': review.get('author', {}).get('playtime_forever', 0) / 60,  # Convert minutes to hours
                'votes_up': review.get('votes_up', 0)  # Include number of upvotes
            })

        # Get the cursor for the next page
        cursor = data.get('cursor', '')
        print(f"Cursor for next page: {cursor}")

        # Check if the cursor repeats
        if cursor in seen_cursors:
            print(f"Repeated cursor detected on page {page}: {cursor}. Stopping pagination.")
            break
        seen_cursors.add(cursor)

        if not cursor:
            print("No cursor provided; ending pagination.")
            break

        time.sleep(1)  # Avoid rate limits

    return reviews


def get_steam_game_info(app_id):
    store_url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&cc=us"
    store_response = requests.get(store_url)
    
    try:
        response_json = store_response.json()
        store_data = response_json.get(str(app_id), {}).get('data', None)
    except (ValueError, AttributeError) as e:
        print(f"Error parsing JSON for app ID {app_id}: {e}")
        print(f"Raw response content: {store_response.text}")
        return None

    # If no data, print and return None
    if not store_data:
        print(f"No data found for app ID {app_id}.")
        print(f"Raw response content: {store_response.text}")
        return None

    # Extract name and price information
    game_name = store_data.get('name', 'N/A')
    price = store_data.get('price_overview', {}).get('final_formatted', 'Free')  # Check 'final_formatted' in 'price_overview'

    # Fetch and store review details
    reviews = get_steam_reviews(app_id)

    # Compile the results
    return {
        'appid': app_id,
        'name': game_name,
        'price': price,
        'reviews': reviews
    }


# Collecting all game data
def collect_game_data(appid_list):
    all_games_info = []

    for app_id in appid_list:
        print(f"Fetching data for appid: {app_id}")
        game_info = get_steam_game_info(app_id)
        if game_info:
            all_games_info.append(game_info)
        time.sleep(1)

    return all_games_info

# Save the data to a file
def save_game_info_to_file(all_games_info, output_file='data/steam_reviews_data.csv'):
    # Flatten the data into one row per game, with review lists as columns
    flattened_data = []

    for game in all_games_info:
        flattened_data.append({
            'appid': game['appid'],
            'name': game['name'],
            'price': game['price'],
            'reviews': [review['comment'] for review in game['reviews']],
            'review_dates': [review['date'] for review in game['reviews']],
            'recommended_status': [review['recommended'] for review in game['reviews']],
            'playtime_hours': [review['playtime_hours'] for review in game['reviews']]
        })

    # Convert to DataFrame
    df = pd.DataFrame(flattened_data)
    df.to_csv(output_file, index=False)
    print(f"File saved as {output_file}.")

# Execute data collection and saving
all_games_info = collect_game_data([1222670])
save_game_info_to_file(all_games_info)


# Getting the dates information so the data can be matched with stock data based on time

In [ ]:
import pandas as pd
df = pd.read_csv('data/steam_reviews_data.csv')
df.head()

In [ ]:
import ast

first_row_dates_str = df.iloc[0, 4]
first_row_dates = ast.literal_eval(first_row_dates_str)
first_row_dates_converted = [pd.to_datetime(date, format='%Y-%m-%d').date() for date in first_row_dates]

oldest_date = min(first_row_dates_converted)

# print(first_row_dates_converted)
print("oldest date: " + str(oldest_date)) 

In [ ]:
first_row_recommendation_str = df.iloc[0, 5]
first_row_recommendation = ast.literal_eval(first_row_recommendation_str)
first_row_recommendation

In [ ]:
from datetime import datetime, timedelta

def generate_date_list(start_date, end_date):
    date_list = [start_date + timedelta(days=i) 
                 for i in range((end_date - start_date).days + 1)]
    return date_list

date_to_index = {date: idx for idx, date in enumerate(dates_list)}
recommend = [0] * len(dates_list)
not_recommend = [0] * len(dates_list)

for i in range(len(first_row_recommendation)):
    index = date_to_index.get(first_row_dates_converted[i])
    if index is None:
        print(f"Date {first_row_dates_converted[i]} not in dates_list, skipping...")
        continue

    if first_row_recommendation[i] == "Recommended":
        recommend[index] += 1
    elif first_row_recommendation[i] == "Not Recommended":
        not_recommend[index] += 1
    else:
        print(f"Unexpected recommendation value '{first_row_recommendation[i]}' at index {i}")

print(recommend)
print(not_recommend)

In [ ]:
print(first_row_dates_converted[0:10])

In [ ]:
first_row_recommendation[0:10]

In [ ]:
from datetime import date

# Convert all datetime.datetime to datetime.date
normalized_dates = [d.date() if isinstance(d, datetime) else d for d in first_row_dates_converted]

# Find the index
index = normalized_dates.index(date(2024, 8, 22))
